In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('synthetic_logs.csv')
df.head(5) 

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert


In [3]:
df.source.unique()

<StringArray>
[      'ModernCRM', 'AnalyticsEngine',        'ModernHR',   'BillingSystem',
   'ThirdPartyAPI',       'LegacyCRM']
Length: 6, dtype: str

In [4]:
df.target_label.unique()

<StringArray>
[        'HTTP Status',      'Critical Error',      'Security Alert',
               'Error', 'System Notification',      'Resource Usage',
         'User Action',      'Workflow Error', 'Deprecation Warning']
Length: 9, dtype: str

In [6]:
%pip install sentence-transformers

  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/739.8 kB ? eta -:--:--
   -------------- ------------------------- 262.1/739.8 kB ? eta -:--:--
   ---------------------------------------- 739.8/739.8 kB 1.8 MB/s  0:00:00
   ---------------------------------------- 0.0/798.3 kB ? eta -:--:--
   ------------- -------------------------- 262.1/798.3 kB ? eta -:--:--
   -------------------------- ------------- 524.3/798.3 kB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 798.3/798.3 kB 1.5 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.0 MB 1.9 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/4.0 MB 1.9 MB/s eta 0:00:02
   --------------- ------------------------ 1.6/4.0 MB 1.9 MB/s eta 0:00:02
   -------------------- ------------

In [7]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['log_message'].tolist())



c:\Users\Moolya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Moolya\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Moolya\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to

In [10]:
dbscan = DBSCAN(eps=0.2, min_samples=1, metric='euclidean')
clusters = dbscan.fit_predict(embeddings)

df['cluster'] = clusters


In [11]:
df.head()

,timestamp,source,log_message,target_label,complexity,cluster
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,3


In [12]:
df.cluster.unique()

array([   0,    1,    2, ..., 1061, 1062, 1063], dtype=int64)

In [13]:
cluster_counts=df['cluster'].value_counts()
largest_clusters = cluster_counts[cluster_counts > 10].index

for cluster in largest_clusters:
    cluster_logs = df[df['cluster'] == cluster]
    print(f"Cluster {cluster}:")
    print(cluster_logs['log_message'].head(5).to_string(index=False))
    print("\n")

Cluster 0:
nova.osapi_compute.wsgi.server [req-b9718cd8-f6...
nova.osapi_compute.wsgi.server [req-4895c258-b2...
nova.osapi_compute.wsgi.server [req-d4f8d0c2-4f...
nova.osapi_compute.wsgi.server [req-6fe0e366-f2...
nova.osapi_compute.wsgi.server [req-945d1f31-a2...


Cluster 4:
nova.osapi_compute.wsgi.server [req-f0bffbc3-5a...
nova.osapi_compute.wsgi.server [req-5e6e042b-f9...
nova.osapi_compute.wsgi.server [req-4d05bae9-8a...
nova.osapi_compute.wsgi.server [req-9174a757-01...
nova.osapi_compute.wsgi.server [req-b2ffcdcc-26...


Cluster 3:
nova.osapi_compute.wsgi.server [req-ee8bc8ba-92...
nova.osapi_compute.wsgi.server [req-2bf7cfee-a2...
nova.osapi_compute.wsgi.server [req-5f1c2027-e1...
nova.osapi_compute.wsgi.server [req-4e83daf7-a2...
nova.osapi_compute.wsgi.server [req-fe9ef402-d3...


Cluster 12:
Backup completed successfully.
Backup completed successfully.
Backup completed successfully.
Backup completed successfully.
Backup completed successfully.


Cluster 50:
nova.metadata.w

In [20]:
classify_with_regex("User User123 logged OUT.")

'User Action'

In [21]:
classify_with_regex("Hey how are you?")

In [23]:
df['regex_label'] = df['log_message'].apply(classify_with_regex)
df[df.regex_label.notnull()][['log_message', 'regex_label']].head(10)

,log_message,regex_label
7,File data_6169.csv uploaded successfully by us...,System Notification
14,File data_3847.csv uploaded successfully by us...,System Notification
15,Backup completed successfully.,System Notification
18,Account with ID 5351 created by User634.,User Action
27,User User685 logged out.,User Action
30,Backup started at 2025-05-14 07:06:55.,System Notification
36,System reboot initiated by user User243.,System Notification
44,Backup started at 2025-02-15 20:00:19.,System Notification
48,File data_7366.csv uploaded successfully by us...,System Notification
50,System updated to version 3.9.1.,System Notification


In [24]:
df_non_regex = df[df['regex_label'].isnull()].copy()
df_non_regex

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0,NaN
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,NaN
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2,NaN
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0,NaN
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,3,NaN
...,...,...,...,...,...,...,...
2405,2025-08-13 07:29:25,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert,4,NaN
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert,1061,NaN
2407,2025-08-03 03:07:47,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert,172,NaN
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert,1062,NaN


In [25]:
print(df_non_regex['target_label'].value_counts()[df_non_regex['target_label'].value_counts() <= 5].index.tolist())

['Workflow Error', 'Deprecation Warning']


In [28]:
df_non_legacy = df_non_regex[df_non_regex.source != 'LegacyCRM']

In [29]:
filtered_embeddings = model.encode(df_non_legacy['log_message'].tolist())


In [31]:
X = filtered_embeddings
y = df_non_legacy['target_label']
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(filtered_embeddings, y, test_size=0.3, random_state=42)

logistic_model = LogisticRegression(max_iter=1000,random_state=42)
logistic_model.fit(X_train, y_train)

y_pred = logistic_model.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

                precision    recall  f1-score   support

Critical Error       0.91      1.00      0.95        48
         Error       0.98      0.89      0.93        47
   HTTP Status       1.00      1.00      1.00       304
Resource Usage       1.00      1.00      1.00        49
Security Alert       1.00      0.99      1.00       123

      accuracy                           0.99       571
     macro avg       0.98      0.98      0.98       571
  weighted avg       0.99      0.99      0.99       571



In [32]:
%pip install joblib

Note: you may need to restart the kernel to use updated packages.


In [33]:
import joblib

joblib.dump(logistic_model, 'logistic_model.pkl')

['logistic_model.pkl']